## LG Aimers 9기 — 강한 기준선 재구축 Forward CV (Colab)

기존 05~08 결과를 불러오지 않고 독립적으로 기준선을 다시 만든다. `season`, `game_type`을 입력에서 제거하고, pre-2023 F 경기의 누적 `asof_*` 오염 보정, 최근 시즌 가중, 학습 데이터 추세 기반 prior shift, 4개 LightGBM 구조 × 5개 seed를 순서대로 검증한다. 이 노트북은 제출 ZIP을 만들지 않는다.

### 0. Drive 마운트와 프로젝트 확인

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import time
import zipfile

if Path('/content').exists():
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

candidates = [
    Path('/content/drive/MyDrive/LG_aimers_9th'),
    Path('/content/drive/MyDrive/LG_Aimers_9th'),
    Path('/content/drive/MyDrive/LG Aimers 9th'),
    Path.cwd(),
    Path.cwd().parent,
]
required_relative = [
    Path('data/train.csv'),
    Path('src/features.py'),
    Path('src/f_contamination.py'),
    Path('src/evaluation.py'),
    Path('src/metrics.py'),
    Path('experiments/09_strong_baseline_cv.py'),
]
PROJECT_ROOT = next(
    (path for path in candidates if all((path / rel).exists() for rel in required_relative)),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        '프로젝트 폴더 또는 09에 필요한 experiments/src 파일을 찾지 못했습니다.'
    )
PROJECT_ROOT = PROJECT_ROOT.resolve()
print('PROJECT_ROOT:', PROJECT_ROOT)
print('train.csv size (GB):', round((PROJECT_ROOT / 'data/train.csv').stat().st_size / 1e9, 3))

### 1. 실행 환경 확인

기본 PyPI LightGBM은 CPU를 사용한다. A100 런타임에서도 실행 가능하며 높은 시스템 메모리가 유리하지만 GPU 자체는 사용하지 않는다. Colab 기본 NumPy·pandas는 다시 설치하지 않고 LightGBM만 맞춘다.

In [ ]:
install = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '-q', 'lightgbm==4.6.0'],
    capture_output=True,
    text=True,
)
if install.returncode != 0:
    print(install.stdout[-3000:])
    print(install.stderr[-3000:])
    raise RuntimeError('LightGBM 설치에 실패했습니다.')

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

gpu_binary = shutil.which('nvidia-smi')
gpu_name = 'not detected'
if gpu_binary:
    gpu = subprocess.run(
        [gpu_binary, '--query-gpu=name,memory.total', '--format=csv,noheader'],
        capture_output=True, text=True, check=False,
    )
    gpu_name = gpu.stdout.strip() or 'detected but unavailable'
print('Python:', sys.version.split()[0])
print('LightGBM:', lgb.__version__)
print('pandas:', pd.__version__, '| NumPy:', np.__version__)
print('CPU threads:', os.cpu_count(), '| GPU:', gpu_name)

### 2. 실험 계약 고정

점수를 본 뒤 조건을 바꾸지 않도록 학습·보정·선택 규칙을 먼저 고정한다. 2023과 2024 어느 fold도 원시 기준선보다 나빠지지 않는 후보만 안전 후보로 인정한다.

In [ ]:
experiment_contract = pd.DataFrame([
    ['검증', 'train < validation season인 2023·2024 forward fold'],
    ['입력 제외', 'season, game_type'],
    ['ID', 'fold train vocabulary로만 인코딩한 pitcher_id, batter_id 포함'],
    ['F 보정', 'fold train에서 복원한 reverse 증분만 사용; validation은 lookup만'],
    ['보정식', 'pitcher reverse += D/n, pitcher·batter success -= D/n'],
    ['시간 가중', 'half-life 0.5 season'],
    ['prior', 'fold train의 시즌별 target rate 선형 추세를 한 시즌 외삽'],
    ['앙상블', '4개 LightGBM 구조 × 5개 seed'],
    ['결합', 'probability mean과 logit mean을 함께 기록'],
    ['안전 게이트', 'pooled 개선 + 2023·2024 Brier 모두 raw_single 이하'],
    ['제출', '생성하지 않음; CV 결과 확인 후 별도 최종 학습'],
], columns=['item', 'fixed rule'])
display(experiment_contract)

### 3. 실행 설정

첫 실행은 CPU LightGBM 44개를 학습한다. 완료된 fold/member는 Drive에 캐시되므로 런타임이 끊기면 같은 셀을 다시 실행한다.

In [ ]:
VALID_SEASONS = [2023, 2024]
THREADS = max(1, min(8, os.cpu_count() or 2))
N_ESTIMATORS = 320
LEARNING_RATE = 0.015
HALF_LIFE = 0.5
SEEDS = [42, 7, 2024, 1234, 31337]
VARIANTS = ['base', 'wide', 'extra', 'lowff']
FORCE_RERUN = False
OUTPUT_ROOT = PROJECT_ROOT / 'outputs/strong_baseline_cv_v1'

settings = pd.DataFrame([
    ['valid seasons', VALID_SEASONS],
    ['threads', THREADS],
    ['estimators / learning rate', f'{N_ESTIMATORS} / {LEARNING_RATE}'],
    ['half-life', HALF_LIFE],
    ['seeds', SEEDS],
    ['variants', VARIANTS],
    ['ensemble members', len(SEEDS) * len(VARIANTS)],
    ['force rerun', FORCE_RERUN],
], columns=['setting', 'value'])
display(settings)

### 4. 강한 기준선 Forward CV 실행

In [ ]:
command = [
    sys.executable,
    str(PROJECT_ROOT / 'experiments/09_strong_baseline_cv.py'),
    '--train-path', str(PROJECT_ROOT / 'data/train.csv'),
    '--output-dir', str(OUTPUT_ROOT),
    '--valid-seasons', *map(str, VALID_SEASONS),
    '--threads', str(THREADS),
    '--n-estimators', str(N_ESTIMATORS),
    '--learning-rate', str(LEARNING_RATE),
    '--half-life', str(HALF_LIFE),
    '--seeds', *map(str, SEEDS),
    '--variants', *VARIANTS,
]
if FORCE_RERUN:
    command.append('--force-rerun')
print('Running:', ' '.join(command))
started = time.perf_counter()
subprocess.run(command, check=True)
print('elapsed minutes:', round((time.perf_counter() - started) / 60, 1))

### 5. 최신 run과 무결성 확인

In [ ]:
with (OUTPUT_ROOT / 'latest_run.json').open(encoding='utf-8') as handle:
    latest = json.load(handle)
recorded_run = Path(latest['run_dir'])
RUN_DIR = next(
    (path for path in [recorded_run, OUTPUT_ROOT / recorded_run.name] if path.exists()),
    None,
)
if RUN_DIR is None:
    raise FileNotFoundError('09 최신 run 폴더를 찾지 못했습니다.')
with (RUN_DIR / 'summary.json').open(encoding='utf-8') as handle:
    summary = json.load(handle)
with (RUN_DIR / 'integrity_audit.json').open(encoding='utf-8') as handle:
    integrity = json.load(handle)
if integrity['status'] != 'passed':
    raise RuntimeError('09 무결성 검사를 통과하지 못했습니다.')
fold_checks = []
for season, fold in integrity['folds'].items():
    fold_checks.append({
        'valid_season': int(season),
        'train_rows': fold['train_rows'],
        'valid_rows': fold['valid_rows'],
        'max_train_season': fold['max_train_season'],
        'season excluded': fold['season_feature_excluded'],
        'game_type excluded': fold['game_type_feature_excluded'],
        'members': fold['member_count'],
        'probabilities finite': fold['all_probabilities_finite'],
    })
display(pd.DataFrame(fold_checks))
print('signature:', summary['signature'])
print('RUN_DIR:', RUN_DIR)

### 6. F 누적 오염 복원·보정 감사

`success_target_agreement`는 누적 성공률 증분이 실제 행 target과 맞는지 확인한다. `q_*`는 각 시즌의 R reverse 비율에서 F reverse 비율을 뺀 train-only 누락 추정치다.

In [ ]:
with (RUN_DIR / 'contamination_audit.json').open(encoding='utf-8') as handle:
    contamination = json.load(handle)
contam_rows = []
for season, fold in contamination.items():
    artifact = fold['artifact']
    recovery = artifact['recovery_audit']
    valid_transform = fold['valid_transform']
    row = {
        'valid_season': int(season),
        'recovery_fraction': recovery['reverse_recovery_fraction'],
        'success_target_agreement': recovery['success_target_agreement'],
        'dirty_rows': recovery['dirty_rows'],
        'valid pitcher adj mean': valid_transform['pitcher_adjustment_mean'],
        'valid batter adj mean': valid_transform['batter_adjustment_mean'],
        'row independence': fold['row_independence']['status'],
    }
    row.update({f'q_{year}': value for year, value in artifact['q_by_season'].items()})
    contam_rows.append(row)
contamination_table = pd.DataFrame(contam_rows).sort_values('valid_season')
display(contamination_table)
assert contamination_table['success_target_agreement'].min() >= 0.999
assert contamination_table['row independence'].eq('passed').all()

### 7. Train-only prior 추세 확인

`validation_target_rate_diagnostic_only`는 학습이 끝난 OOF 해석용이며 prior 산출에는 사용되지 않는다. forecast와 실제의 차이가 fold마다 같은 방향인지 확인한다.

In [ ]:
with (RUN_DIR / 'prior_audit.json').open(encoding='utf-8') as handle:
    priors = json.load(handle)
prior_rows = []
for season, item in priors.items():
    prior_rows.append({
        'valid_season': int(season),
        'unweighted train prior': item['unweighted_train_prior'],
        'recency-weighted source prior': item['recency_weighted_train_prior'],
        'train-trend forecast': item['forecast'],
        'actual diagnostic': item['validation_target_rate_diagnostic_only'],
        'forecast gap': item['forecast'] - item['validation_target_rate_diagnostic_only'],
        'trend slope': item['slope'],
    })
display(pd.DataFrame(prior_rows).sort_values('valid_season'))

### 8. 단계별 성능과 누적 기여

Brier와 delta는 낮을수록 좋고 BSS는 높을수록 좋다. `delta_*_x1e6`가 음수면 원시 기준선보다 개선이다.

In [ ]:
stage_summary = pd.read_csv(RUN_DIR / 'stage_summary.csv')
stage_summary['delta_pooled_brier_vs_raw_x1e6'] = (
    1e6 * stage_summary['delta_pooled_brier_vs_raw']
)
stage_summary['delta_brier_2023_vs_raw_x1e6'] = (
    1e6 * stage_summary['delta_brier_2023_vs_raw']
)
stage_summary['delta_brier_2024_vs_raw_x1e6'] = (
    1e6 * stage_summary['delta_brier_2024_vs_raw']
)
display(stage_summary[[
    'stage', 'pooled_brier', 'pooled_bss_raw', 'pooled_mean_gap',
    'brier_2023', 'brier_2024',
    'delta_pooled_brier_vs_raw_x1e6',
    'delta_brier_2023_vs_raw_x1e6', 'delta_brier_2024_vs_raw_x1e6',
    'passes_anchor_gate',
]])

plt.figure(figsize=(11, 4))
colors = np.where(stage_summary['passes_anchor_gate'], '#2ca02c', '#9aa0a6')
plt.bar(stage_summary['stage'], stage_summary['pooled_bss_raw'], color=colors)
plt.ylabel('Pooled raw BSS (higher is better)')
plt.xticks(rotation=40, ha='right')
plt.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.show()

### 9. Fold별 전체·R·F 안정성

pooled 하나만 보지 않고 2023·2024와 R/F가 같은 방향인지 확인한다.

In [ ]:
stage_metrics = pd.read_csv(RUN_DIR / 'stage_metrics.csv')
fold_metrics = stage_metrics[stage_metrics['partition'].eq('fold')].copy()
brier_by_fold = fold_metrics.pivot_table(
    index='stage', columns=['valid_season', 'scope'], values='brier'
)
display(brier_by_fold.loc[stage_summary['stage']])

raw = fold_metrics[fold_metrics['stage'].eq('raw_single')][
    ['valid_season', 'scope', 'brier']
].rename(columns={'brier': 'raw_brier'})
delta = fold_metrics.merge(raw, on=['valid_season', 'scope'], validate='many_to_one')
delta['delta_brier_x1e6'] = 1e6 * (delta['brier'] - delta['raw_brier'])
delta_plot = delta[delta['scope'].isin(['R', 'F'])].pivot_table(
    index='stage', columns=['valid_season', 'scope'], values='delta_brier_x1e6'
)
display(delta_plot.loc[stage_summary['stage']])

### 10. 앙상블 멤버 단독 성능과 다양성

In [ ]:
member_metrics = pd.read_csv(RUN_DIR / 'member_metrics.csv')
member_pivot = member_metrics.pivot_table(
    index='member', columns='valid_season', values=['brier', 'bss_raw']
)
display(member_pivot.sort_values(('brier', 2024)).head(20))

member_corr = pd.read_csv(RUN_DIR / 'member_correlations.csv')
display(member_corr.sort_values('pearson_vs_logit_ensemble').head(20))
print('member count:', len(member_corr))

### 11. 선택 기준선의 취약 구간

선택 모델이 raw 대비 좋아졌더라도 특정 count·손잡이·F 구간을 훼손하면 다음 피처 실험의 우선 대상이 된다.

In [ ]:
selected_stage = summary['selected_stage']
segments = pd.read_csv(RUN_DIR / 'segment_metrics.csv')
focus = segments[segments['stage'].isin(['raw_single', selected_stage])].copy()
wide = focus.pivot_table(
    index=['valid_season', 'group', 'value'], columns='stage', values='brier'
).reset_index()
wide['selected_minus_raw_x1e6'] = 1e6 * (
    wide[selected_stage] - wide['raw_single']
)
display(wide.sort_values('selected_minus_raw_x1e6').head(20))
print('가장 악화된 구간')
display(wide.sort_values('selected_minus_raw_x1e6', ascending=False).head(20))

### 12. 피처 중요도 확인

중요도는 인과가 아니라 다음 ablation 후보를 고르는 진단값이다.

In [ ]:
importance = pd.read_csv(RUN_DIR / 'feature_importance.csv')
ensemble_importance = importance[
    importance['member'].str.startswith('corrected_recency__')
].copy()
importance_summary = (
    ensemble_importance.groupby('feature', observed=True)['importance']
    .agg(mean_gain='mean', median_gain='median', max_gain='max')
    .sort_values('mean_gain', ascending=False)
    .reset_index()
)
display(importance_summary.head(40))

### 13. 자동 판정과 다음 단계

In [ ]:
decision = pd.DataFrame([
    ['selected stage', summary['selected_stage']],
    ['selection status', summary['selection_status']],
    ['raw pooled Brier', summary['raw_pooled_brier']],
    ['selected pooled Brier', summary['selected_pooled_brier']],
    ['selected pooled BSS raw', summary['selected_pooled_bss_raw']],
    ['selected Brier 2023', summary['selected_brier_2023']],
    ['selected Brier 2024', summary['selected_brier_2024']],
    ['ensemble members', summary['member_count']],
    ['integrity', summary['integrity_status']],
], columns=['item', 'value'])
display(decision)
if summary['selected_stage'] == 'raw_single':
    print('판정: 누적 변경이 두 fold 안전 게이트를 통과하지 못했다. 제출 학습으로 넘어가지 않는다.')
else:
    print('판정: 강한 기준선 후보를 확보했다. 결과 검토 후 독립 피처·모델 실험 또는 최종 학습으로 연결한다.')

### 14. 검토용 결과 ZIP 생성

실행 완료 노트북과 `strong09_review.zip`을 전달한다. 멤버 재조합이나 행별 분석이 필요하면 `oof_predictions.csv.gz`도 함께 전달한다.

In [ ]:
review_names = [
    'summary.json',
    'configuration.json',
    'integrity_audit.json',
    'contamination_audit.json',
    'prior_audit.json',
    'stage_summary.csv',
    'stage_metrics.csv',
    'segment_metrics.csv',
    'member_metrics.csv',
    'member_correlations.csv',
    'member_training.json',
    'feature_importance.csv',
]
REVIEW_ZIP = OUTPUT_ROOT / 'strong09_review.zip'
with zipfile.ZipFile(REVIEW_ZIP, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for name in review_names:
        path = RUN_DIR / name
        if not path.exists():
            raise FileNotFoundError(path)
        archive.write(path, arcname=name)
OOF_PATH = RUN_DIR / 'oof_predictions.csv.gz'
print('검토 ZIP:', REVIEW_ZIP)
print('검토 ZIP 크기 (MB):', round(REVIEW_ZIP.stat().st_size / 1e6, 2))
print('OOF:', OOF_PATH)
print('OOF 크기 (MB):', round(OOF_PATH.stat().st_size / 1e6, 2))
print('전달: 실행된 09_strong_baseline_cv.ipynb + strong09_review.zip + oof_predictions.csv.gz')